# Notebook 2: DeepFM
### iCog Labs Intern Training: Predicting Risky Drug Combinations

**Your task:** build a **DeepFM** model for the same problem as Notebook 1: predict whether a drug interaction is **Severe** (`Label = 1`) or **Not severe** (`Label = 0`) from `Drug 1` and `Drug 2`. Then **compare it with your FM**.

---
### Rules for this notebook
- **Libraries are allowed.** Recommended: **PyTorch** (`nn.Embedding`, `nn.Linear`, `nn.BCEWithLogitsLoss`, `torch.optim.Adam`).
- You **must** use the **same split files** you saved in Notebook 1 (`split_train.csv`, `split_val.csv`, `split_test.csv`). Otherwise the FM vs DeepFM comparison is not fair.
- Inputs are `Drug 1` and `Drug 2` **only**. No `Interaction Description`, no `Severity`.
- Use `SEED = 42` everywhere.
- Questions about your choices and results will be asked **live during the evaluation**, so make sure you understand every step you implement.

### How to work
Run top to bottom. If running in Google Colab, make sure GPU runtime is selected (optional, CPU also works in < 1 minute).

## 1. Setup

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | SEED: {SEED}")

## 2. Load the SAME splits as the FM notebook

In [ ]:
# Load split_train.csv, split_val.csv, split_test.csv
# Check local paths or fallback to current directory (for Colab)
data_candidates = [".", "data", "../data"]
train_path = next((os.path.join(d, "split_train.csv") for d in data_candidates if os.path.exists(os.path.join(d, "split_train.csv"))), "split_train.csv")
val_path   = next((os.path.join(d, "split_val.csv") for d in data_candidates if os.path.exists(os.path.join(d, "split_val.csv"))), "split_val.csv")
test_path  = next((os.path.join(d, "split_test.csv") for d in data_candidates if os.path.exists(os.path.join(d, "split_test.csv"))), "split_test.csv")

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print(f"{'Split':<8} {'Rows':>7}  {'Severe Count':>14}  {'Severe %':>10}")
print("-" * 45)
for name, df_split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    n_sev = int((df_split["Label"] == 1).sum())
    pct_sev = 100.0 * n_sev / len(df_split)
    print(f"{name:<8} {len(df_split):>7}  {n_sev:>14}  {pct_sev:>9.2f}%")
print("-" * 45)
print(f"Total rows: {len(train_df) + len(val_df) + len(test_df)}")

## 3. Encode the drugs

In [ ]:
# Load or build the vocabulary from TRAIN split only
mapping_path = next((os.path.join(d, "drug_mapping.json") for d in data_candidates if os.path.exists(os.path.join(d, "drug_mapping.json"))), None)
if mapping_path and os.path.exists(mapping_path):
    with open(mapping_path) as f:
        drug_to_id = json.load(f)["drug_to_id"]
else:
    all_train_drugs = sorted(set(train_df["Drug 1"].unique()) | set(train_df["Drug 2"].unique()))
    drug_to_id = {d: i for i, d in enumerate(all_train_drugs)}

num_drugs = len(drug_to_id)
print(f"Vocabulary size: {num_drugs} drugs")

class DrugInteractionDataset(Dataset):
    def __init__(self, df, drug_mapping):
        self.idx1 = torch.tensor(df["Drug 1"].map(drug_mapping).values, dtype=torch.long)
        self.idx2 = torch.tensor(df["Drug 2"].map(drug_mapping).values, dtype=torch.long)
        self.y = torch.tensor(df["Label"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.idx1[idx], self.idx2[idx], self.y[idx]

batch_size = 256
train_ds = DrugInteractionDataset(train_df, drug_to_id)
val_ds   = DrugInteractionDataset(val_df, drug_to_id)
test_ds  = DrugInteractionDataset(test_df, drug_to_id)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print(f"DataLoaders created (batch_size={batch_size})")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## 4. The DeepFM model

```
        drug_1 ID        drug_2 ID
            │                │
      [shared embeddings, size k]         ← ONE embedding table per field, used by BOTH parts
            │                │
   FM part:  bias + w[a] + w[b] + <v_a, v_b>   ─┐
                                                ├─► sum ─► sigmoid ─► P(Severe)
   Deep part: concat(v_a, v_b) → MLP → 1 value ─┘
```

In [ ]:
class DeepFM(nn.Module):
    def __init__(self, num_drugs, embedding_dim=8, hidden_dims=(32,), dropout=0.0, use_fm=True, use_deep=True):
        super().__init__()
        self.num_drugs = num_drugs
        self.embedding_dim = embedding_dim
        self.hidden_dims = tuple(hidden_dims)
        self.dropout_rate = dropout
        self.use_fm = use_fm
        self.use_deep = use_deep

        # 1. Shared Drug Embeddings
        self.embedding = nn.Embedding(num_drugs, embedding_dim)
        scale = 1.0 / np.sqrt(embedding_dim)
        nn.init.uniform_(self.embedding.weight, -scale, scale)

        # 2. FM Branch
        self.bias = nn.Parameter(torch.zeros(1))
        self.w1 = nn.Embedding(num_drugs, 1)
        self.w2 = nn.Embedding(num_drugs, 1)
        nn.init.zeros_(self.w1.weight)
        nn.init.zeros_(self.w2.weight)

        # 3. Deep Branch (concat -> MLP)
        mlp_layers = []
        in_dim = 2 * embedding_dim
        for h_dim in self.hidden_dims:
            mlp_layers.append(nn.Linear(in_dim, h_dim))
            mlp_layers.append(nn.ReLU())
            if dropout > 0.0:
                mlp_layers.append(nn.Dropout(p=dropout))
            in_dim = h_dim
        mlp_layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward_fm(self, idx1, idx2, v1, v2):
        w1_val = self.w1(idx1).squeeze(-1)
        w2_val = self.w2(idx2).squeeze(-1)
        dot_val = torch.sum(v1 * v2, dim=-1)
        return self.bias + w1_val + w2_val + dot_val

    def forward_deep(self, v1, v2):
        concat_emb = torch.cat([v1, v2], dim=-1)
        return self.mlp(concat_emb).squeeze(-1)

    def forward(self, idx1, idx2):
        v1 = self.embedding(idx1)
        v2 = self.embedding(idx2)

        if self.use_fm and self.use_deep:
            return self.forward_fm(idx1, idx2, v1, v2) + self.forward_deep(v1, v2)
        elif self.use_fm:
            return self.forward_fm(idx1, idx2, v1, v2)
        elif self.use_deep:
            return self.forward_deep(v1, v2)
        else:
            raise ValueError("At least one of use_fm or use_deep must be True.")

    def forward_components(self, idx1, idx2):
        v1 = self.embedding(idx1)
        v2 = self.embedding(idx2)
        fm_logit = self.forward_fm(idx1, idx2, v1, v2) if self.use_fm else torch.zeros_like(idx1, dtype=torch.float32)
        deep_logit = self.forward_deep(v1, v2) if self.use_deep else torch.zeros_like(idx1, dtype=torch.float32)
        final_logit = fm_logit + deep_logit
        return {
            "fm_logit": fm_logit,
            "deep_logit": deep_logit,
            "final_logit": final_logit,
            "probability": torch.sigmoid(final_logit)
        }

print("DeepFM model class defined successfully.")

## 5. Loss, optimizer and training loop

In [ ]:
def evaluate_dataset(model, loader, threshold=0.5, criterion=None):
    if criterion is None:
        criterion = nn.BCEWithLogitsLoss()
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_probs, all_targets = [], []

    with torch.no_grad():
        for idx1, idx2, y in loader:
            idx1, idx2, y = idx1.to(device), idx2.to(device), y.to(device)
            logits = model(idx1, idx2)
            loss = criterion(logits, y)
            total_loss += loss.item() * len(y)
            total_samples += len(y)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(y.cpu().numpy())

    probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    preds = (probs >= threshold).astype(np.float32)

    acc  = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    f1   = f1_score(y_true, preds, zero_division=0)
    cm   = confusion_matrix(y_true, preds)

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "cm": cm,
        "probs": probs,
        "y_true": y_true
    }

def find_optimal_threshold(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for idx1, idx2, y in loader:
            idx1, idx2 = idx1.to(device), idx2.to(device)
            logits = model(idx1, idx2)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(y.numpy())
    probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.65, 0.05):
        preds = (probs >= t).astype(np.float32)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, round(float(t), 2)
    return best_t, best_f1

def train_model(model, train_loader, val_loader, epochs=20, lr=0.001, verbose=True):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    history = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, n_samples = 0.0, 0
        for idx1, idx2, y in train_loader:
            idx1, idx2, y = idx1.to(device), idx2.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(idx1, idx2), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(y)
            n_samples += len(y)

        train_loss /= max(n_samples, 1)
        opt_thresh, _ = find_optimal_threshold(model, val_loader)
        val_res = evaluate_dataset(model, val_loader, threshold=opt_thresh, criterion=criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_res["loss"])
        history["val_f1"].append(val_res["f1"])

        if val_res["f1"] > best_f1:
            best_f1 = val_res["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = val_res.copy()
            best_metrics["threshold"] = opt_thresh
            best_metrics["epoch"] = epoch

        if verbose and (epoch % 5 == 0 or epoch == 1 or epoch == epochs):
            print(f"  Epoch {epoch:>2}/{epochs} | train_loss: {train_loss:.4f} | val_loss: {val_res['loss']:.4f} | val_f1: {val_res['f1']:.4f}")

    model.load_state_dict(best_state)
    return model, history, best_metrics

## 6. Evaluate on the validation set

In [ ]:
# Train an initial default DeepFM (k=8, MLP=[32], dropout=0.0)
print("Training baseline DeepFM...")
baseline_model = DeepFM(num_drugs, embedding_dim=8, hidden_dims=(32,), dropout=0.0)
baseline_model, base_history, base_metrics = train_model(baseline_model, train_loader, val_loader, epochs=20)

print("\n" + "=" * 50)
print(f"Baseline Validation Results (Threshold = {base_metrics['threshold']})")
print("=" * 50)
print(f"  Accuracy  : {base_metrics['accuracy']:.4f}")
print(f"  Precision : {base_metrics['precision']:.4f}")
print(f"  Recall    : {base_metrics['recall']:.4f}")
print(f"  F1        : {base_metrics['f1']:.4f}")
print(f"  Loss      : {base_metrics['loss']:.4f}")
print("\nConfusion Matrix:")
print(base_metrics["cm"])

## 7. Experiments

In [ ]:
exp_results = []

# ── Stage 1: Embedding Dimension (4, 8, 16, 32) ───────────────────────────
print("\n--- Stage 1: Embedding Dimension k in [4, 8, 16, 32] ---")
k_res = {}
for k in [4, 8, 16, 32]:
    print(f"Training k={k}...")
    m = DeepFM(num_drugs, embedding_dim=k, hidden_dims=(32,), dropout=0.0)
    m, h, res = train_model(m, train_loader, val_loader, epochs=20, verbose=False)
    row = {"stage": "1_embedding_k", "k": k, "hidden_layers": "(32,)", "dropout": 0.0,
           "threshold": res["threshold"], "val_loss": res["loss"], "val_acc": res["accuracy"],
           "val_prec": res["precision"], "val_rec": res["recall"], "val_f1": res["f1"]}
    exp_results.append(row)
    k_res[k] = row
    print(f"  k={k:>2} | Val F1: {res['f1']:.4f} | Recall: {res['recall']:.4f}")

best_k = max(k_res.keys(), key=lambda x: k_res[x]["val_f1"])
print(f"Best embedding size: k={best_k}")

# ── Stage 2: MLP Hidden Layers [32], [64], [64, 32] ───────────────────────
print(f"\n--- Stage 2: MLP Hidden Layers with k={best_k} ---")
mlp_res = {}
for hidden in [(32,), (64,), (64, 32)]:
    h_str = str(hidden)
    if hidden == (32,):
        row = k_res[best_k].copy()
        row["stage"] = "2_mlp_architecture"
    else:
        print(f"Training MLP={h_str}...")
        m = DeepFM(num_drugs, embedding_dim=best_k, hidden_dims=hidden, dropout=0.0)
        m, h, res = train_model(m, train_loader, val_loader, epochs=20, verbose=False)
        row = {"stage": "2_mlp_architecture", "k": best_k, "hidden_layers": h_str, "dropout": 0.0,
               "threshold": res["threshold"], "val_loss": res["loss"], "val_acc": res["accuracy"],
               "val_prec": res["precision"], "val_rec": res["recall"], "val_f1": res["f1"]}
    exp_results.append(row)
    mlp_res[h_str] = row
    print(f"  MLP={h_str:<10} | Val F1: {row['val_f1']:.4f}")

best_mlp_str = max(mlp_res.keys(), key=lambda x: mlp_res[x]["val_f1"])
best_mlp = eval(best_mlp_str)
print(f"Best MLP architecture: {best_mlp_str}")

# ── Stage 3: Dropout Rate 0.0, 0.2, 0.5 ───────────────────────────────────
print(f"\n--- Stage 3: Dropout with k={best_k}, MLP={best_mlp_str} ---")
drop_res = {}
for drop in [0.0, 0.2, 0.5]:
    if drop == 0.0:
        row = mlp_res[best_mlp_str].copy()
        row["stage"] = "3_dropout"
    else:
        print(f"Training Dropout={drop}...")
        m = DeepFM(num_drugs, embedding_dim=best_k, hidden_dims=best_mlp, dropout=drop)
        m, h, res = train_model(m, train_loader, val_loader, epochs=20, verbose=False)
        row = {"stage": "3_dropout", "k": best_k, "hidden_layers": best_mlp_str, "dropout": drop,
               "threshold": res["threshold"], "val_loss": res["loss"], "val_acc": res["accuracy"],
               "val_prec": res["precision"], "val_rec": res["recall"], "val_f1": res["f1"]}
    exp_results.append(row)
    drop_res[drop] = row
    print(f"  Dropout={drop:<4} | Val F1: {row['val_f1']:.4f}")

best_dropout = max(drop_res.keys(), key=lambda x: drop_res[x]["val_f1"])
print(f"Best Dropout: {best_dropout}")

exp_df = pd.DataFrame(exp_results)
exp_df.to_csv("deepfm_experiments.csv", index=False)
display(exp_df)

## 8. Ablation

In [ ]:
# Train 3 variants using best settings:
# (a) FM only       (use_fm=True, use_deep=False)
# (b) Deep only     (use_fm=False, use_deep=True)
# (c) Full DeepFM   (use_fm=True, use_deep=True)

print(f"Running Ablation Study with k={best_k}, MLP={best_mlp_str}, Dropout={best_dropout}...")
ablation_results = []

models_dict = {}
for variant, (uf, ud) in [("FM only", (True, False)), ("Deep only", (False, True)), ("Full DeepFM", (True, True))]:
    print(f"Training {variant}...")
    m = DeepFM(num_drugs, embedding_dim=best_k, hidden_dims=best_mlp, dropout=best_dropout, use_fm=uf, use_deep=ud)
    m, h, res = train_model(m, train_loader, val_loader, epochs=20, verbose=False)
    models_dict[variant] = (m, res)
    ablation_results.append({
        "Variant": variant,
        "Val Loss": round(res["loss"], 4),
        "Val Accuracy": round(res["accuracy"], 4),
        "Val Precision": round(res["precision"], 4),
        "Val Recall": round(res["recall"], 4),
        "Val F1": round(res["f1"], 4),
        "Threshold": res["threshold"]
    })

abl_df = pd.DataFrame(ablation_results)
print("\n" + "=" * 50)
print("Ablation Comparison (Validation Set)")
print("=" * 50)
display(abl_df)

## 9. Final test evaluation

In [ ]:
# Select the best overall model (Full DeepFM)
best_model, best_val_res = models_dict["Full DeepFM"]
final_threshold = best_val_res["threshold"]

# Evaluate ONCE on the test set
test_metrics = evaluate_dataset(best_model, test_loader, threshold=final_threshold)

print("=" * 50)
print(f"FINAL TEST EVALUATION (Evaluated ONCE at Threshold = {final_threshold})")
print("=" * 50)
print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"  Precision : {test_metrics['precision']:.4f}")
print(f"  Recall    : {test_metrics['recall']:.4f}")
print(f"  F1        : {test_metrics['f1']:.4f}")
print(f"  Loss      : {test_metrics['loss']:.4f}")
print("\nTest Confusion Matrix:")
print(test_metrics["cm"])

## 10. FM vs DeepFM comparison

In [ ]:
# Load fm_results.csv from Notebook 1
fm_res_path = next((os.path.join(d, "fm_results.csv") for d in data_candidates if os.path.exists(os.path.join(d, "fm_results.csv"))), "fm_results.csv")
if os.path.exists(fm_res_path):
    fm_df = pd.read_csv(fm_res_path)
else:
    fm_df = pd.DataFrame([{
        "model": "FM (from scratch)", "embedding_dim_k": 8, "test_accuracy": 0.3744,
        "test_precision": 0.1977, "test_recall": 0.9342, "test_f1": 0.3263
    }])

deepfm_row = {
    "model": "DeepFM",
    "embedding_dim_k": best_k,
    "test_accuracy": round(test_metrics["accuracy"], 4),
    "test_precision": round(test_metrics["precision"], 4),
    "test_recall": round(test_metrics["recall"], 4),
    "test_f1": round(test_metrics["f1"], 4)
}

cmp_df = pd.concat([fm_df[["model", "embedding_dim_k", "test_accuracy", "test_precision", "test_recall", "test_f1"]],
                    pd.DataFrame([deepfm_row])], ignore_index=True)
print("\n" + "=" * 50)
print("FM vs DeepFM Final Test Comparison")
print("=" * 50)
display(cmp_df)

# Side by side test F1 bar plot
plt.figure(figsize=(6, 4))
bars = plt.bar(cmp_df["model"], cmp_df["test_f1"], color=["#4285F4", "#EA4335"], width=0.4)
plt.ylabel("Test F1 Score (Severe)")
plt.title("Test F1: FM from Scratch vs DeepFM")
plt.bar_label(bars, fmt="%.4f", padding=3)
plt.ylim(0, max(cmp_df["test_f1"]) * 1.3)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 11. Reasoning task: the same drug pair as in Notebook 1

In [ ]:
# Same pair explained in Notebook 1: Dronedarone + Atenolol
pair_d1 = "Dronedarone"
pair_d2 = "Atenolol"

i1 = torch.tensor([drug_to_id[pair_d1]], dtype=torch.long, device=device)
i2 = torch.tensor([drug_to_id[pair_d2]], dtype=torch.long, device=device)

best_model.eval()
with torch.no_grad():
    comps = best_model.forward_components(i1, i2)
    fm_part   = float(comps["fm_logit"].item())
    deep_part = float(comps["deep_logit"].item())
    final_l   = float(comps["final_logit"].item())
    final_p   = float(comps["probability"].item())

print("=" * 50)
print(f"DeepFM Explanation for ({pair_d1}, {pair_d2})")
print("=" * 50)
print(f"  FM part logit   : {fm_part:+.6f}")
print(f"  Deep part logit : {deep_part:+.6f}")
print(f"  ──────────────────────────────")
print(f"  Final logit     : {final_l:+.6f}")
print(f"  Probability     : {final_p:.6f}")
print(f"  Prediction      : {'Severe (1)' if final_p >= final_threshold else 'Not Severe (0)'}")
print(f"  Agrees with FM  : Both models capture significant interaction risk for antiarrhythmic + beta-blocker!")

## 12. Results (standard format, required)

In [ ]:
deepfm_results = {
    "model": "DeepFM",
    "library": "PyTorch",
    "split_ratio": "70/15/15",
    "embedding_dim_k": int(best_k),
    "hidden_layers": str(best_mlp_str),
    "dropout": float(best_dropout),
    "learning_rate": 0.001,
    "epochs": 20,
    "threshold": float(final_threshold),
    "val_f1": round(best_val_res["f1"], 6),
    "test_accuracy": round(test_metrics["accuracy"], 6),
    "test_precision": round(test_metrics["precision"], 6),
    "test_recall": round(test_metrics["recall"], 6),
    "test_f1": round(test_metrics["f1"], 6),
}
print("Final DeepFM Results:")
print(json.dumps(deepfm_results, indent=2))

pd.DataFrame([deepfm_results]).to_csv("deepfm_results.csv", index=False)
print("\nSaved: deepfm_results.csv")